In [ ]:
%load_ext autoreload
%autoreload 2
from happytransformer import HappyTextToText
from neuspell import BertChecker
from transformers import BartForConditionalGeneration, BartTokenizer, pipeline, T5ForConditionalGeneration, T5Tokenizer

from benchmark import ModelBenchmark
from helpers import get_data_from_file


In [ ]:
CHECKPOINTS = "./checkpoints/"
corrupt, clean = get_data_from_file('test')
pred_func_simple = lambda model, text: model(text)[0]['generated_text']
pred_func_vennify = lambda model, data: model.generate_text(f"grammar: {data}").text
pred_func_grammarly = lambda model, text: model(f"Fix grammatical errors: {text}")[0]['generated_text']
benchmark = ModelBenchmark(verbose=True)

In [ ]:
model_path = CHECKPOINTS + "grammarly-coedit-large-finetuned"
model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                   local_files_only=True)
tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=2048)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_grammarly, start_idx=666, num_sen=10)

In [ ]:
model_path = CHECKPOINTS + "pszemraj-bart-base-grammar-synthesis-finetuned"
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                     local_files_only=True)
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_simple, start_idx=23745, num_sen=30)

In [ ]:
model_path = CHECKPOINTS + "oliverguhr-spelling-correction-english-base-finetuned"
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                     local_files_only=True)
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=2048)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_simple, start_idx=23745, num_sen=30)

In [ ]:
model_path = "./checkpoints/vennify-t5-base-grammar-correction-finetuned"
happy_tt = HappyTextToText(model_type="T5", model_name=model_path)
res = benchmark.get_wrong_words(happy_tt, corrupt, clean, pred_func_vennify, start_idx=666, num_sen=10)

In [ ]:
model_path = CHECKPOINTS + "prithivida-grammar_error_correcter_v1-finetuned"
model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                   local_files_only=True)
tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=2048)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_simple, start_idx=666, num_sen=10)

In [ ]:
path = "checkpoints/subwordbert-probwordnoise/finetuned_model"
checker = BertChecker(device="cuda")
checker.from_pretrained(path)


In [ ]:
pred_func = lambda model, data: model.correct_string(data, correct_spaces=True)
res = benchmark.get_wrong_words(checker, corrupt, clean, pred_func, tokenizer=None, start_idx=666, num_sen=2000)